# Section 3.1: Duplicate Handling - Integration Example

This notebook shows how to integrate the production pipeline into YOUR existing notebook.

**Before:** Hardcoded duplicate checks
**After:** Configuration-driven pipeline with audit logging

## BEFORE: Your Original Code

```python
# Exact row duplicates
data.duplicated().sum()
data[data.duplicated()]

# Customer-level duplicates check
data[data['customer_id'].duplicated(keep=False)].sort_values('customer_id')
(data.groupby('customer_id').size().loc[lambda x: x > 1])
data[data.duplicated()]

# Inspect if any exist
data[data.duplicated(keep=False)]

# Drop the duplicates
data = data.drop_duplicates()

# Check that drop worked
duplicates_remaining = int(data.duplicated().sum())
print(f'There are {duplicates_remaining} duplicate(s) remaining')
```

**Problem:** Ad-hoc, not reusable, hard to test, no audit trail

## AFTER: Pipeline Version

Replace all that with 3 cells:

### Cell 1: Import and Configure

In [ ]:
import pandas as pd
from typing import Dict, Tuple, Any
import logging

# Configure logging for audit trail
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

# ==============================================================================
# DUPLICATE HANDLING CONFIGURATION
# ==============================================================================

DUPLICATE_CONFIG = {
    'check_exact_rows': True,
    'check_customer_level': True,
    'customer_id_column': 'customer_id',
    'keep': 'first',
    'fail_if_duplicates_remain': True,
    'subset': None,
}

### Cell 2: Define the Function

In [ ]:
def handle_duplicates(df: pd.DataFrame, config: Dict[str, Any] = None) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Detect and remove duplicate records at both exact-row and customer-level.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe to check for duplicates
    config : Dict
        Configuration dict (see DUPLICATE_CONFIG above)
    
    Returns:
    --------
    df_clean : pd.DataFrame
        Dataframe with duplicates removed
    audit : Dict
        Audit trail with detailed logging
    """
    
    if config is None:
        config = DUPLICATE_CONFIG
    
    # Initialize audit trail
    audit = {
        'status': 'STARTED',
        'rows_input': len(df),
        'rows_removed': 0,
        'checks_performed': [],
        'duplicate_details': {},
        'errors': []
    }
    
    df_clean = df.copy()
    
    # =========================================================================
    # CHECK 1: EXACT ROW DUPLICATES
    # =========================================================================
    if config['check_exact_rows']:
        n_exact_dupes = df_clean.duplicated(subset=config['subset'], keep=False).sum()
        audit['checks_performed'].append('exact_row_check')
        
        if n_exact_dupes > 0:
            dup_mask = df_clean.duplicated(subset=config['subset'], keep=False)
            dup_rows = df_clean[dup_mask].sort_values(
                by=list(df_clean.columns[:5])
            ).reset_index(drop=True)
            
            audit['duplicate_details']['exact_duplicates'] = {
                'count': n_exact_dupes,
                'duplicate_pairs': len(df_clean[dup_mask]) // 2,
                'sample_indices': dup_rows.index.tolist()[:6]
            }
            
            logger.info(f"✓ Exact row check: Found {n_exact_dupes} duplicate rows")
            logger.info(f"  Sample of duplicates (first 6 rows):")
            logger.info(dup_rows.head(6))
        else:
            audit['duplicate_details']['exact_duplicates'] = {'count': 0}
            logger.info(f"✓ Exact row check: No exact duplicates found")
    
    # =========================================================================
    # CHECK 2: CUSTOMER-LEVEL DUPLICATES
    # =========================================================================
    if config['check_customer_level'] and config['customer_id_column'] in df_clean.columns:
        customer_col = config['customer_id_column']
        n_unique_customers = df_clean[customer_col].nunique()
        n_total_rows = len(df_clean)
        n_customer_dupes = n_total_rows - n_unique_customers
        
        audit['checks_performed'].append('customer_level_check')
        
        if n_customer_dupes > 0:
            dup_customer_mask = df_clean[customer_col].duplicated(keep=False)
            dup_customers = df_clean[dup_customer_mask].sort_values(customer_col)
            dupes_per_customer = df_clean[customer_col].value_counts()
            dupes_per_customer = dupes_per_customer[dupes_per_customer > 1]
            
            audit['duplicate_details']['customer_level'] = {
                'duplicate_customers': len(dupes_per_customer),
                'duplicate_rows': n_customer_dupes,
                'distribution': dupes_per_customer.to_dict(),
            }
            
            logger.info(f"✓ Customer-level check: {len(dupes_per_customer)} customers appear multiple times")
            logger.info(f"  Total rows involved: {n_customer_dupes}")
            logger.info(f"  Distribution: {dict(dupes_per_customer)}")
            logger.info(f"  Sample of duplicated customers:")
            logger.info(dup_customers.head(8))
        else:
            audit['duplicate_details']['customer_level'] = {
                'duplicate_customers': 0,
                'duplicate_rows': 0
            }
            logger.info(f"✓ Customer-level check: No customer-level duplicates found")
    
    # =========================================================================
    # REMOVAL: DROP DUPLICATES
    # =========================================================================
    rows_before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=config['subset'], keep=config['keep'])
    rows_removed = rows_before - len(df_clean)
    
    audit['rows_removed'] = int(rows_removed)
    audit['rows_output'] = len(df_clean)
    
    if rows_removed > 0:
        logger.info(f"\n✓ Duplicate removal: {rows_removed} rows removed")
        logger.info(f"  Input rows: {rows_before} → Output rows: {len(df_clean)}")
    else:
        logger.info(f"\n✓ No duplicates to remove")
    
    # =========================================================================
    # VALIDATION: Confirm no duplicates remain
    # =========================================================================
    remaining_exact_dupes = df_clean.duplicated(subset=config['subset']).sum()
    
    if remaining_exact_dupes > 0:
        error_msg = f"Duplicate removal failed: {remaining_exact_dupes} duplicates remain"
        audit['errors'].append(error_msg)
        audit['status'] = 'FAILED'
        logger.error(f"✗ VALIDATION FAILED: {error_msg}")
        
        if config['fail_if_duplicates_remain']:
            raise ValueError(error_msg)
    else:
        audit['status'] = 'SUCCESS'
        logger.info(f"\n✓ VALIDATION PASSED: No duplicates remain")
    
    logger.info(f"\n" + "="*70)
    logger.info(f"DUPLICATE HANDLING SUMMARY")
    logger.info(f"="*70)
    logger.info(f"Status: {audit['status']}")
    logger.info(f"Rows removed: {audit['rows_removed']}")
    logger.info(f"Input shape: ({audit['rows_input']}, {df.shape[1]})")
    logger.info(f"Output shape: ({audit['rows_output']}, {df_clean.shape[1]})")
    logger.info(f"="*70)
    
    return df_clean, audit

### Cell 3: Run the Pipeline

This is where your actual data processing happens.

In [ ]:
# Load your data (replace with your actual file path)
# data = pd.read_csv('your_loan_data.csv')

# Run the duplicate handling pipeline
data_cleaned, audit_trail = handle_duplicates(data, config=DUPLICATE_CONFIG)

# The audit trail shows everything that happened
print(f"\nAudit Trail Summary:")
print(f"  Status: {audit_trail['status']}")
print(f"  Rows removed: {audit_trail['rows_removed']}")
print(f"  Input rows: {audit_trail['rows_input']}")
print(f"  Output rows: {audit_trail['rows_output']}")

### Cell 4: Inspect Results

In [ ]:
# Optional: Detailed inspection of what was removed
print("\n" + "="*70)
print("DETAILED DUPLICATE DETAILS")
print("="*70)

for check_type, details in audit_trail['duplicate_details'].items():
    print(f"\n{check_type}:")
    for key, value in details.items():
        print(f"  {key}: {value}")

print(f"\n" + "="*70)
print(f"Cleaned data shape: {data_cleaned.shape}")
print(f"No duplicates remain: {data_cleaned.duplicated().sum() == 0}")

### Cell 5: Drop customer_id and Continue

Now that we've verified duplicates at the customer level, we can drop the column.

In [ ]:
# customer_id has served its purpose (duplicate checking)
# Drop it since it provides little use for modeling
data_cleaned = data_cleaned.drop(columns=['customer_id'])

print(f"Columns after dropping customer_id:")
print(data_cleaned.columns.tolist())
print(f"\nShape: {data_cleaned.shape}")
print(f"\nFirst few rows:")
print(data_cleaned.head())

## Comparison: Before vs After

| Aspect | Before | After |
|--------|--------|-------|
| **Lines of code** | 10+ exploratory lines | 3 clean cells |
| **Reusability** | One-off script | Reusable function |
| **Audit trail** | Manual inspection | Automatic logging |
| **Configuration** | Hardcoded | Configuration dict |
| **Error handling** | Basic | Comprehensive validation |
| **Professional appearance** | Ad-hoc | Production-ready |

---

## To a Recruiter, This Shows:

✅ **Software engineering best practices:**
- Configuration-driven code
- Modular, reusable functions
- Comprehensive error handling

✅ **Data quality mindset:**
- Checks before removing data
- Audit trail for reproducibility
- Validation at each step

✅ **Production readiness:**
- Logging for debugging
- Metrics for monitoring
- Clear documentation

✅ **Professional communication:**
- Clear function docstrings
- Meaningful variable names
- Structured output